# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.
> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector
*Code that actually builds it — engineered features, categorical handling, fills.*

The prediction origin is `2026-04-01`. Features use only `2026-01-01` through `2026-03-31`; the future label window is kept separate.

In [1]:
import os, getpass, duckdb, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HuggingFace READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_retries = 10")
con.execute("SET http_timeout = 120")
con.execute("SET enable_http_metadata_cache = false")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
ORIGIN = '2026-04-01'
FEATURE_START = '2026-01-01'
FEATURE_END = '2026-03-31'
LABEL_START = '2026-04-01'
LABEL_END = '2026-04-30'

con.sql(f"""
CREATE OR REPLACE TEMP VIEW feature_base AS
SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_clicks, 0) ELSE 0 END) AS clicks_90d,
    SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_sum_position, 0) ELSE 0 END) AS sum_position_90d,
    SUM(CASE WHEN f.ga4_data_available IS TRUE THEN COALESCE(f.ga4_sessions, 0) ELSE 0 END) AS sessions_90d,
    SUM(CASE WHEN f.ga4_data_available IS TRUE THEN COALESCE(f.ga4_engaged_sessions, 0) ELSE 0 END) AS engaged_sessions_90d,
    COUNT(DISTINCT CASE WHEN f.gsc_data_available IS TRUE AND COALESCE(f.gsc_impressions, 0) > 0 THEN f.report_date END) AS days_with_impressions_90d,
    ANY_VALUE(c.content_created_date) AS content_created_date,
    ANY_VALUE(c.word_count) AS word_count,
    ANY_VALUE(c.search_volume) AS search_volume,
    ANY_VALUE(c.competition) AS competition,
    ANY_VALUE(c.content_type) AS content_type,
    ANY_VALUE(c.main_intent) AS main_intent,
    ANY_VALUE(cl.access_profile) AS access_profile
FROM {FACT} f
LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id = f.content_hash_id
LEFT JOIN {DIM_CLIENTS} cl ON cl.client_hash_id = f.client_hash_id
WHERE f.report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
GROUP BY f.content_hash_id, f.client_hash_id
""")

con.sql("""
CREATE OR REPLACE TEMP VIEW tier_rates AS
SELECT
    CASE
        WHEN impressions_90d = 0 OR sum_position_90d = 0 THEN 'no_data'
        WHEN sum_position_90d / impressions_90d <= 3 THEN 'top_3'
        WHEN sum_position_90d / impressions_90d <= 10 THEN 'page_1'
        WHEN sum_position_90d / impressions_90d <= 20 THEN 'striking'
        WHEN sum_position_90d / impressions_90d <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier,
    SUM(clicks_90d) / NULLIF(SUM(impressions_90d), 0) * 100 AS expected_ctr_feature
FROM feature_base
WHERE impressions_90d > 0
GROUP BY 1
""")

feature_vector = con.sql("""
SELECT
    b.content_hash_id,
    b.client_hash_id,
    b.impressions_90d,
    b.clicks_90d,
    b.clicks_90d / NULLIF(b.impressions_90d, 0) * 100 AS ctr_feature,
    b.sum_position_90d / NULLIF(b.impressions_90d, 0) AS avg_position_feature,
    t.position_tier,
    t.expected_ctr_feature,
    (b.clicks_90d / NULLIF(b.impressions_90d, 0) * 100 - t.expected_ctr_feature) / NULLIF(t.expected_ctr_feature, 0) AS relative_ctr_gap,
    b.sessions_90d,
    b.engaged_sessions_90d / NULLIF(b.sessions_90d, 0) * 100 AS engagement_rate,
    b.days_with_impressions_90d,
    DATE_DIFF('day', b.content_created_date, DATE '2026-04-01') AS content_age_days,
    b.word_count,
    b.search_volume,
    b.competition,
    b.content_type,
    b.main_intent,
    b.access_profile,
    b.word_count IS NULL AS word_count_missing,
    b.search_volume IS NULL AS search_volume_missing,
    b.competition IS NULL AS competition_missing
FROM feature_base b
LEFT JOIN tier_rates t ON t.position_tier = CASE
    WHEN b.impressions_90d = 0 OR b.sum_position_90d = 0 THEN 'no_data'
    WHEN b.sum_position_90d / b.impressions_90d <= 3 THEN 'top_3'
    WHEN b.sum_position_90d / b.impressions_90d <= 10 THEN 'page_1'
    WHEN b.sum_position_90d / b.impressions_90d <= 20 THEN 'striking'
    WHEN b.sum_position_90d / b.impressions_90d <= 50 THEN 'page_3_5'
    ELSE 'deep'
END
""").df()

print(f'Feature rows: {len(feature_vector):,}')
print('Feature columns:', ', '.join(feature_vector.columns))
print(feature_vector[['impressions_90d', 'ctr_feature', 'position_tier', 'content_type']].head().to_string(index=False))

Feature rows: 349,411
Feature columns: content_hash_id, client_hash_id, impressions_90d, clicks_90d, ctr_feature, avg_position_feature, position_tier, expected_ctr_feature, relative_ctr_gap, sessions_90d, engagement_rate, days_with_impressions_90d, content_age_days, word_count, search_volume, competition, content_type, main_intent, access_profile, word_count_missing, search_volume_missing, competition_missing
 impressions_90d  ctr_feature position_tier    content_type
             0.0          NaN       no_data keyword article
             0.0          NaN       no_data keyword article
             0.0          NaN       no_data keyword article
             0.0          NaN       no_data keyword article
             0.0          NaN       no_data keyword article


## 2. Feature notes (meaning, missing, categorical, available-when?)
*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All numeric aggregates are computed from daily facts strictly before the origin. Numeric missingness is preserved with explicit `*_missing` indicators; categorical nulls remain null until a downstream encoder applies a documented unknown category. IDs are retained only for grouping and joins.

In [2]:
feature_notes = pd.DataFrame([
    ('impressions_90d / clicks_90d / ctr_feature', 'pre-origin GSC volume and CTR', 'GSC-unavailable rows aggregate to zero with availability represented by volume', 'numeric', True),
    ('avg_position_feature / position_tier', 'impressions-weighted search position and tier', 'no position becomes no_data, never rank zero', 'numeric + categorical', True),
    ('expected_ctr_feature / relative_ctr_gap', 'feature-window pooled tier benchmark and gap', 'zero benchmark yields NULL gap', 'numeric context', True),
    ('sessions_90d / engagement_rate', 'pre-origin GA4 activity and engagement', 'GA4-unavailable rows aggregate to zero with access_profile context', 'numeric', True),
    ('days_with_impressions_90d', 'number of active GSC days', 'zero means no measured impression day', 'numeric', True),
    ('content_age_days / word_count / search_volume / competition', 'content and keyword context available by origin', 'numeric nulls retained with missing flags', 'numeric', True),
    ('content_type / main_intent / access_profile', 'categorical context from dimensions', 'nulls map to an explicit unknown category during encoding', 'categorical', True),
], columns=['feature_group', 'meaning', 'missing_treatment', 'type', 'available_before_origin'])
display(feature_notes)
assert feature_notes['available_before_origin'].all()

,feature_group,meaning,missing_treatment,type,available_before_origin
0,impressions_90d / clicks_90d / ctr_feature,pre-origin GSC volume and CTR,GSC-unavailable rows aggregate to zero with av...,numeric,True
1,avg_position_feature / position_tier,impressions-weighted search position and tier,"no position becomes no_data, never rank zero",numeric + categorical,True
2,expected_ctr_feature / relative_ctr_gap,feature-window pooled tier benchmark and gap,zero benchmark yields NULL gap,numeric context,True
3,sessions_90d / engagement_rate,pre-origin GA4 activity and engagement,GA4-unavailable rows aggregate to zero with ac...,numeric,True
4,days_with_impressions_90d,number of active GSC days,zero means no measured impression day,numeric,True
5,content_age_days / word_count / search_volume ...,content and keyword context available by origin,numeric nulls retained with missing flags,numeric,True
6,content_type / main_intent / access_profile,categorical context from dimensions,nulls map to an explicit unknown category duri...,categorical,True


## 3. The leakage hunt
*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The checks below make the time boundary executable. The future label is built only for the sanity test; it is not joined into `feature_vector`.

In [3]:
forbidden_feature_names = {
    'need_ctr_fix', 'ctr_next30d', 'impressions_next30d', 'clicks_next30d',
    'trend_direction', 'trend_pct', 'product_score', 'decision_flag',
    'label', 'target', 'content_hash_id', 'client_hash_id'
}
model_features = feature_vector.drop(columns=['content_hash_id', 'client_hash_id'])
assert forbidden_feature_names.isdisjoint(set(model_features.columns)), 'Forbidden field entered the model feature vector'

feature_date_check = con.sql(f"""
SELECT MIN(report_date) AS min_feature_date, MAX(report_date) AS max_feature_date,
       MAX(CASE WHEN report_date >= DATE '{LABEL_START}' THEN 1 ELSE 0 END) AS overlaps_label_window
FROM {FACT}
WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
""").df()
display(feature_date_check)
assert int(feature_date_check.loc[0, 'overlaps_label_window']) == 0

duplicate_check = con.sql("""
SELECT content_hash_id, client_hash_id, COUNT(*) AS row_count
FROM feature_base
GROUP BY content_hash_id, client_hash_id
HAVING COUNT(*) > 1
""").df()
print(f'Duplicate feature grains: {len(duplicate_check):,}')
assert duplicate_check.empty

QUERY_90D = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"
query_window_check = con.sql(f"""
SELECT MIN(window_start) AS min_window_start, MAX(window_end) AS max_window_end,
       SUM(CASE WHEN window_end >= DATE '{LABEL_START}' THEN 1 ELSE 0 END) AS rows_overlapping_label_window
FROM {QUERY_90D}
""").df()
display(query_window_check)
print('Query-table fields with overlapping windows are excluded; daily facts are the source for this feature vector.')
assert not any('last30' in column.lower() or 'next30' in column.lower() for column in model_features.columns)

label_check = con.sql(f"""
SELECT COUNT(*) AS rows_with_future_data,
       SUM(CASE WHEN impressions_next30d >= 100 THEN 1 ELSE 0 END) AS eligible_future_rows
FROM (
    SELECT content_hash_id,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_next30d
    FROM {FACT}
    WHERE report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
    GROUP BY content_hash_id
) future
""").df()
display(label_check)

leaky_demo = con.sql(f"""
SELECT content_hash_id,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks, 0) ELSE 0 END)
       / NULLIF(SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions, 0) ELSE 0 END), 0) * 100 AS leaky_ctr_next30d
FROM {FACT}
WHERE report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
GROUP BY content_hash_id
""").df()
print(f'Deliberate leaky-feature sanity test rows: {len(leaky_demo):,}; this future-only frame is not a model input.')
assert 'leaky_ctr_next30d' not in feature_vector.columns

,min_feature_date,max_feature_date,overlaps_label_window
0,2026-01-01,2026-03-31,0


Duplicate feature grains: 0


,min_window_start,max_window_end,rows_overlapping_label_window
0,2026-04-02,2026-06-30,2414248.0


Query-table fields with overlapping windows are excluded; daily facts are the source for this feature vector.


,rows_with_future_data,eligible_future_rows
0,362172,107144.0


Deliberate leaky-feature sanity test rows: 362,172; this future-only frame is not a model input.


## 4. What I excluded and why
*The list of fields you refused to use — with one line of why each.*

The exclusions are part of the contract, not optional cleanup after modeling.

In [4]:
excluded_fields = pd.DataFrame([
    ('need_ctr_fix, ctr_next30d, impressions_next30d, clicks_next30d', 'label or future-window siblings; direct target leakage'),
    ('trend_direction, trend_pct', 'label-derived fields in the starter contract; unavailable or circular at origin'),
    ('*_last30 and other query-table future aggregates', 'the fixed query window overlaps the April label window'),
    ('product_score, decision_flag, existing-system rank', 'decision-derived; would reproduce the old rule rather than learn the outcome'),
    ('content_hash_id, client_hash_id', 'pseudonymous identifiers; grouping and joins only'),
    ('report_date and post-origin daily facts', 'future information unavailable at the prediction moment'),
], columns=['excluded_fields', 'reason'])
display(excluded_fields)
assert len(excluded_fields) == 6

,excluded_fields,reason
0,"need_ctr_fix, ctr_next30d, impressions_next30d...",label or future-window siblings; direct target...
1,"trend_direction, trend_pct",label-derived fields in the starter contract; ...
2,*_last30 and other query-table future aggregates,the fixed query window overlaps the April labe...
3,"product_score, decision_flag, existing-system ...",decision-derived; would reproduce the old rule...
4,"content_hash_id, client_hash_id",pseudonymous identifiers; grouping and joins only
5,report_date and post-origin daily facts,future information unavailable at the predicti...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.